# Chapitre 9 - Exercices 9.1 à 9.3 : lisseurs linéaires et backfitting

Ces trois exercices étudient le backfitting (algorithme 9.1 du livre) sous l'angle de
l'**algèbre linéaire** : quand les lisseurs sont linéaires, $\hat{\mathbf f}_j = S_j\,
\mathbf r$, l'algorithme devient une méthode itérative classique de résolution d'un
système linéaire.

- **Exercice 9.1** : une spline de lissage préserve la partie linéaire de l'ajustement ;
  conséquence pour le recentrage dans le backfitting.
- **Exercice 9.2** : le backfitting est un algorithme de Gauss–Seidel par blocs ;
  convergence pour deux lisseurs symétriques.
- **Exercice 9.3** : avec des projections orthogonales, les équations du backfitting
  sont les équations normales des moindres carrés.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl.ch09_arbres import (SplineLisseur, MatriceLisseur, local_linear_matrix,
                             backfitting)

set_style()
rng = np.random.RandomState(0)

## Exercice 9.1 : La spline de lissage préserve la partie linéaire

**Énoncé (reformulé).** Décomposons la réponse en $y_i = \hat y_i + r_i$, où $\hat y$
est l'ajustement de la régression linéaire simple de $y$ sur $x$ et $r$ le résidu.
Soit $S$ la matrice de lissage d'une spline de lissage. Montrer que
$$S y = \hat y + S r,$$
c'est-à-dire que la spline reproduit exactement la partie linéaire. Montrer qu'il en va
de même pour la régression linéaire locale (section 6.1.1). En déduire que l'étape de
recentrage (seconde ligne de l'étape 2 de l'algorithme 9.1) est inutile.

### Rappels

- **Spline de lissage** (section 5.4) : $\hat f$ minimise $\sum_i(y_i-f(x_i))^2 +
  \lambda\int f''^2$. Dans une base $B_1,\dots,B_K$ de splines contenant les fonctions
  affines, $f=\sum_k c_kB_k$, on a $\hat c = (B^TB+\lambda\Omega)^{-1}B^Ty$ et
  $S=B(B^TB+\lambda\Omega)^{-1}B^T$, avec $\Omega_{kl}=\int B_k''B_l''$.
- **Régression linéaire locale** (éq. (6.7)-(6.9)) : en chaque point $x_0$, on résout
  $\min_{\alpha,\beta}\sum_i K_\lambda(x_0,x_i)\,(y_i-\alpha-\beta x_i)^2$ et l'on pose
  $\hat f(x_0)=\hat\alpha(x_0)+\hat\beta(x_0)x_0$. Matriciellement, avec $\mathbf B$ la
  matrice $N\times 2$ de lignes $(1,x_i)$ et $W(x_0)=\mathrm{diag}(K_\lambda(x_0,x_i))$ :
  $\hat f(x_0) = (1,x_0)\,(\mathbf B^TW(x_0)\mathbf B)^{-1}\mathbf B^TW(x_0)\,y$.

### Solution : le cas de la spline de lissage

Le vecteur $\hat y$ est de la forme $\hat y_i = a + b x_i$ : ce sont les valeurs aux
$x_i$ de la fonction affine $\ell(x)=a+bx$.

**Étape 1 : $S\hat y = \hat y$.** Appliquons la spline de lissage aux données
$(x_i,\hat y_i)$. Pour la fonction $\ell$ :
- le terme d'ajustement $\sum_i(\hat y_i-\ell(x_i))^2$ vaut $0$ ;
- le terme de pénalité $\lambda\int\ell''^2$ vaut $0$ car $\ell''=0$.

Le critère étant toujours $\ge 0$, $\ell$ en est un minimiseur ; le critère étant
strictement convexe (pour $\lambda>0$ et au moins deux valeurs distinctes de $x$), c'est
**le** minimiseur. Donc la spline ajustée à $\hat y$ est $\ell$ elle-même : $S\hat y =
\hat y$.

**Étape 2 : linéarité.** $S$ est une matrice, donc par linéarité
$$Sy = S(\hat y + r) = S\hat y + Sr = \hat y + Sr. \qquad\blacksquare$$

En d'autres termes, la spline ne touche pas à la partie linéaire de $y$ et ne lisse que
le résidu $r$.

### Solution : le cas de la régression linéaire locale

Même raisonnement : si $y = \hat y$ avec $\hat y_i = a+bx_i$, alors en chaque point
$x_0$ le problème des moindres carrés pondérés locaux admet la solution exacte
$(\alpha,\beta)=(a,b)$ (somme des carrés nulle). La matrice $\mathbf B^TW(x_0)\mathbf B$
étant inversible (au moins deux points de poids non nul), cette solution est unique. Donc
$\hat f(x_0)=a+bx_0$ pour tout $x_0$ : $S\hat y=\hat y$, puis $Sy=\hat y+Sr$ par
linéarité.

Algébriquement : $(\mathbf B^TW\mathbf B)^{-1}\mathbf B^TW\,\mathbf B\,(a,b)^T = (a,b)^T$,
d'où $\hat f(x_0) = (1,x_0)(a,b)^T$.

### Vérification numérique

In [2]:
N = 80
x = np.sort(rng.uniform(0, 1, N))
y = np.sin(5 * x) + 2 * x + 0.3 * rng.randn(N)
A = np.column_stack([np.ones(N), x])
y_lin = A @ np.linalg.lstsq(A, y, rcond=None)[0]
r = y - y_lin

S_spl = SplineLisseur(x, df=6).matrix()        # spline de lissage exacte (nœuds en tous les x_i)
S_ll = local_linear_matrix(x, bandwidth=0.08)  # régression linéaire locale
for nom, S in [("spline de lissage", S_spl), ("linéaire locale", S_ll)]:
    print("%-18s : max |S y - (y_lin + S r)| = %.2e" % (nom, np.abs(S @ y - (y_lin + S @ r)).max()))

spline de lissage  : max |S y - (y_lin + S r)| = 1.43e-10
linéaire locale    : max |S y - (y_lin + S r)| = 5.55e-15


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (nom, S) in zip(axes, [("spline de lissage", S_spl), ("linéaire locale", S_ll)]):
    ax.scatter(x, y, s=10, color=GRAY, label="$y$")
    ax.plot(x, y_lin, color=ORANGE, label=r"partie linéaire $\hat y$")
    ax.plot(x, S @ r, color=COLORS[2], label="$Sr$ (résidu lissé)")
    ax.plot(x, S @ y, color=BLUE, label="$Sy = \\hat y + Sr$")
    ax.set(title=nom, xlabel="$x$")
axes[0].set_ylabel("$y$")
axes[0].legend(fontsize=8)
plt.show()

### Conséquence pour le backfitting

Dans l'algorithme 9.1, on applique $S_j$ à des résidus partiels $\mathbf r$ et l'on
recentre le résultat pour que $\sum_i\hat f_j(x_{ij})=0$. Si $\mathbf r$ est de
moyenne nulle, le recentrage est-il nécessaire ?

Décomposons $\mathbf r = \hat{\mathbf r} + \mathbf e$ (partie linéaire + résidu de la
régression linéaire de $\mathbf r$ sur $x_j$). Alors :
- $\hat{\mathbf r}$ est de moyenne $\bar r = 0$ (une régression avec constante
  reproduit la moyenne) ;
- $\mathbf e$ est orthogonal à $\mathbf 1$ et à $x_j$ (équations normales).

D'après ce qui précède, $S_j\mathbf r = \hat{\mathbf r} + S_j\mathbf e$. Il reste à voir
que $S_j\mathbf e$ est de moyenne nulle.

**Pour la spline de lissage**, $S$ est **symétrique** et $S\mathbf 1 = \mathbf 1$ (les
constantes sont des fonctions affines). Donc
$$\mathbf 1^TS\mathbf e = (S^T\mathbf 1)^T\mathbf e = (S\mathbf 1)^T\mathbf e =
\mathbf 1^T\mathbf e = 0 .$$
Ainsi $\mathbf 1^TS_j\mathbf r = \mathbf 1^T\hat{\mathbf r} + 0 = 0$ : lisser une
réponse de moyenne nulle donne un ajustement de moyenne nulle. Comme $\hat\alpha=\bar y$
et que les $\hat f_k$ sont initialement nulles, tous les résidus partiels sont de moyenne
nulle et, par récurrence, toutes les $\hat f_j$ le restent : **le recentrage est
inutile** (en arithmétique exacte).

> **Remarque.** On peut aussi le voir sur les équations normales de la spline,
> $B^T(y-B\hat c)=\lambda\Omega\hat c$. Si $\ell=Ba$ est affine, sa pénalité est nulle,
> $a^T\Omega a=0$, donc $\Omega a=0$ ($\Omega$ étant symétrique positive). En multipliant
> par $a^T$ : $\ell^T(y-Sy)=\lambda a^T\Omega\hat c=0$. Les résidus d'une spline de
> lissage sont **orthogonaux aux fonctions affines**, en particulier à $\mathbf 1$.

**Pour la régression linéaire locale**, $S$ n'est **pas symétrique** : on a toujours
$S\mathbf 1=\mathbf 1$ (chaque ligne somme à 1), mais en général $\mathbf 1^TS \ne
\mathbf 1^T$, de sorte que $\mathbf 1^TS\mathbf e$ n'a aucune raison d'être nul. La
préservation de la partie linéaire reste vraie, mais elle ne suffit pas à garantir des
ajustements de moyenne nulle : pour ce type de lisseur, le recentrage est réellement
utile (même si l'écart est en général faible).